# Klasifikasi Indikasi Gangguan Hutan dengan Data Sintetis

## 1. Latar Belakang

Pemantauan tutupan hutan tropis secara global mengalami lompatan besar dengan hadirnya sistem peringatan dini deforestasi berbasis satelit, seperti *RADD (Radar Alerts for Detecting Deforestation)* dan *GLAD Alerts*. Sistem ini mendeteksi hilangnya tutupan pohon secara periodik (mingguan) menggunakan citra satelit radar (Sentinel-1) dan optik (Sentinel-2).

Namun, terdapat perbedaan mendasar antara **Mendeteksi Perubahan (*Detection*)** dan **Memperkirakan Penyebab Langsung (*Attribution*)**:
1. **Deteksi Perubahan**: Menjawab pertanyaan *"Di mana dan kapan pohon hilang?"*. Sistem ini menandai piksel-piksel hutan yang mengalami penurunan tutupan vegetasi.
2. **Atribusi Penyebab (*Driver Attribution*)**: Menjawab pertanyaan *"Apa jenis aktivitas atau faktor pemicu langsung yang menyebabkan hilangnya tutupan pohon tersebut?"*. Apakah pembukaan lahan pertanian, aktivitas pertambangan, atau bencana alam (seperti longsor atau banjir bandang)?

Mengetahui penyebab langsung sangat krusial bagi pengelola hutan dan pembuat kebijakan. Respons lapangan untuk menangani pertambangan liar tentu sangat berbeda dengan respons penanggulangan bencana alam.

> **Penegasan Ruang Lingkup Akademik:**  
> Studi kasus ini merupakan **demonstrasi penyederhanaan pembelajaran** menggunakan data tabular sintetis, **bukan reproduksi penuh** dari model deep learning CNN multisensor berskala operasional milik Dr. Bart Slagter, dan **bukan sistem pemantauan satelit waktu-nyata**.

## 2. Tujuan Studi Kasus

1. Memahami konsep atribusi penyebab gangguan hutan melalui pendekatan klasifikasi machine learning multikelas (*multiclass classification*).
2. Membangkitkan dataset tabular sintetis 450 sampel yang merepresentasikan 3 kelas indikasi gangguan hutan (*Pertanian*, *Pertambangan*, *Gangguan Alam*) dengan distribusi agak tidak seimbang (*mildly imbalanced*).
3. Menggunakan 5 fitur biofisik satelit sintetis: NDVI sebelum & sesudah, hamburan balik radar VV sebelum & sesudah (dB), dan luas perubahan (hektare).
4. Melatih model `RandomForestClassifier` ringan dan membandingkannya dengan patokan baseline `DummyClassifier`.
5. Mengevaluasi model menggunakan metrik akurasi, presisi, recall, F1 per kelas, macro F1, serta matriks kebingungan (*confusion matrix*).
6. Mendiskusikan dampak ketidakseimbangan kelas, keterbatasan data sintetis, dan batas etis pemodelan teknis.

## 3. Pertanyaan Analisis

1. Bagaimana distribusi kelima fitur biofisik membedakan ketiga kelas penyebab gangguan hutan?
2. Seberapa baik model Random Forest mampu memprediksi kelas minoritas (*Gangguan Alam*) dibandingkan kelas mayoritas (*Pertanian*)?
3. Mengapa metrik Macro F1 lebih objektif dan representatif daripada Akurasi Global pada kasus dengan ketidakseimbangan kelas?
4. Mengapa model klasifikasi satelit berbasis kecerdasan buatan tidak dapat secara otomatis dipakai untuk menetapkan status legalitas hukum atau motif sosial-ekonomi?

## 4. Data dan Spesifikasi Fitur Sintetis

Dataset dibangkitkan secara sintetis dengan `numpy.random.RandomState(seed=42)` sebanyak **450 sampel**, dengan proporsi kelas agak tidak seimbang (*mildly imbalanced*):
- **Pertanian (*Agriculture*)**: 225 sampel (50%) — kelas mayoritas.
- **Pertambangan (*Mining*)**: 135 sampel (30%) — kelas menengah.
- **Gangguan Alam (*Natural Disturbance*)**: 90 sampel (20%) — kelas minoritas.

### Fitur Masukan (5 Fitur Tabular):
1. `ndvi_sebelum`: Indeks Vegetasi (*Normalized Difference Vegetation Index*) sebelum gangguan (rentang normal hutan lebat: 0.65 – 0.88).
2. `ndvi_sesudah`: Indeks Vegetasi setelah gangguan (mengalami penurunan).
3. `radar_vv_sebelum_db`: Sinyal hamburan balik radar SAR Sentinel-1 polarisasi VV sebelum gangguan dalam satuan decibel (rentang tipikal hutan tropis: -12.5 s.d. -7.5 dB karena hamburan volumetrik kanopi).
4. `radar_vv_sesudah_db`: Hamburan balik radar setelah gangguan (permukaan terbuka/tambang/air menghasilkan nilai dB spesifik).
5. `luas_hektare`: Estimasi ukuran patch area gangguan dalam satuan hektare (ha).

> **Asumsi Simulasi:**  
> Distribusi nilai fitur dibuat saling tumpang tindih (*overlapping*) untuk merefleksikan ketidakpastian dunia nyata. Hubungan fitur-label dalam notebook ini adalah **asumsi simulasi statistik pembelajaran**, bukan temuan empiris lapangan.

## 5. Metode dan Alur Kerja

```
Sintesis Data (450 Sampel, Seed 42)
               │
Stratified Train-Test Split (80:20, 360 train : 90 test)
               │
               ├────────────────────────────────────────┐
               ▼                                        ▼
    RandomForestClassifier                     DummyClassifier (Baseline)
    (n_estimators=100, max_depth=6)            (strategy='most_frequent')
               │                                        │
               └───────────────────┬────────────────────┘
                                   ▼
                    Evaluasi Metrik Data Uji:
    Accuracy | Precision | Recall | Per-class F1 | Macro F1 | Confusion Matrix
```

## 6. Implementasi Python

### 6.1 Import Library dan Bangkitkan Data Sintetis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import os

SEED = 42
np.random.seed(SEED)
rng = np.random.RandomState(SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# Bangkitkan 450 sampel sintetis (225 Pertanian, 135 Pertambangan, 90 Gangguan Alam)
n_agri = 225
n_mine = 135
n_nat = 90

# 1. Pertanian: Penurunan NDVI sedang-tinggi, radar VV menurun sedang, luas bervariasi (skala kecil-menengah)
ndvi_bef_agri = rng.normal(0.78, 0.04, n_agri).clip(0.65, 0.88)
ndvi_aft_agri = rng.normal(0.38, 0.06, n_agri).clip(0.20, 0.55)
radar_bef_agri = rng.normal(-9.5, 0.8, n_agri).clip(-12.0, -7.5)
radar_aft_agri = rng.normal(-13.0, 1.2, n_agri).clip(-16.0, -10.0)
area_agri = rng.lognormal(mean=1.5, sigma=0.6, size=n_agri).clip(0.5, 25.0)

# 2. Pertambangan: Penurunan NDVI sangat drastis (tanah terbuka/lubang tambang), radar VV sangat rendah atau kasar, luas relatif besar
ndvi_bef_mine = rng.normal(0.79, 0.04, n_mine).clip(0.68, 0.88)
ndvi_aft_mine = rng.normal(0.18, 0.04, n_mine).clip(0.10, 0.28)
radar_bef_mine = rng.normal(-9.6, 0.8, n_mine).clip(-12.0, -7.5)
radar_aft_mine = rng.normal(-16.5, 1.5, n_mine).clip(-20.0, -12.5)
area_mine = rng.lognormal(mean=2.6, sigma=0.7, size=n_mine).clip(3.0, 80.0)

# 3. Gangguan Alam (longsor/banjir): NDVI menurun cepat, radar VV sangat basah/air (nilai sangat rendah jika tergenang air atau sedang jika longsor tanah basah), luas heterogen
ndvi_bef_nat = rng.normal(0.76, 0.04, n_nat).clip(0.65, 0.85)
ndvi_aft_nat = rng.normal(0.28, 0.08, n_nat).clip(0.12, 0.45)
radar_bef_nat = rng.normal(-9.7, 0.9, n_nat).clip(-12.0, -7.5)
radar_aft_nat = rng.normal(-14.2, 1.8, n_nat).clip(-19.0, -9.5)
area_nat = rng.lognormal(mean=1.8, sigma=0.8, size=n_nat).clip(0.8, 45.0)

# Gabungkan ke dalam satu DataFrame
data_dict = {
    'ndvi_sebelum': np.concatenate([ndvi_bef_agri, ndvi_bef_mine, ndvi_bef_nat]),
    'ndvi_sesudah': np.concatenate([ndvi_aft_agri, ndvi_aft_mine, ndvi_aft_nat]),
    'radar_vv_sebelum_db': np.concatenate([radar_bef_agri, radar_bef_mine, radar_bef_nat]),
    'radar_vv_sesudah_db': np.concatenate([radar_aft_agri, radar_aft_mine, radar_aft_nat]),
    'luas_hektare': np.concatenate([area_agri, area_mine, area_nat]),
    'kelas_gangguan': (['Pertanian'] * n_agri) + (['Pertambangan'] * n_mine) + (['Gangguan Alam'] * n_nat)
}

df_forest = pd.DataFrame(data_dict)
df_forest = df_forest.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"Dataset Berhasil Dibangkitkan: {df_forest.shape[0]} baris, {df_forest.shape[1]} kolom.")
print("\nDistribusi Kelas Gangguan Hutan:")
print(df_forest['kelas_gangguan'].value_counts())
print("\n5 Baris Pertama:")
print(df_forest.head().round(3))

### 6.2 Visualisasi Eksplorasi Fitur Satelit

Membuat 2 visualisasi untuk mengamati sebaran dan pemisahan antarkelas:
1. Scatter plot *NDVI Sesudah vs Luas Hektare* (skala logaritmik).
2. Boxplot *Radar VV Sesudah (dB)* antar kelas gangguan.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

class_colors = {'Pertanian': '#27ae60', 'Pertambangan': '#d35400', 'Gangguan Alam': '#2980b9'}

# Visualisasi 1: Scatter Plot NDVI Sesudah vs Luas Area
for cls_name, grp in df_forest.groupby('kelas_gangguan'):
    ax1.scatter(grp['ndvi_sesudah'], grp['luas_hektare'],
                label=cls_name, color=class_colors[cls_name],
                alpha=0.75, edgecolors='black', s=55)

ax1.set_yscale('log')
ax1.set_title("Distribusi NDVI Sesudah vs Luas Area Gangguan", fontweight='bold')
ax1.set_xlabel("NDVI Sesudah Gangguan", fontweight='bold')
ax1.set_ylabel("Luas Area (Hektare, Skala Log)", fontweight='bold')
ax1.legend(title="Indikasi Gangguan")
ax1.grid(True, linestyle='--', alpha=0.6)

# Visualisasi 2: Boxplot Hamburan Balik Radar VV Sesudah (dB)
data_radar = [df_forest[df_forest['kelas_gangguan'] == cls]['radar_vv_sesudah_db']
              for cls in ['Pertanian', 'Pertambangan', 'Gangguan Alam']]
bp = ax2.boxplot(data_radar, labels=['Pertanian', 'Pertambangan', 'Gangguan Alam'], patch_artist=True)
for patch, cls in zip(bp['boxes'], ['Pertanian', 'Pertambangan', 'Gangguan Alam']):
    patch.set_facecolor(class_colors[cls])
    patch.set_alpha(0.7)

ax2.set_title("Hamburan Balik Radar Sentinel-1 VV Sesudah (dB)", fontweight='bold')
ax2.set_xlabel("Kelas Gangguan Hutan", fontweight='bold')
ax2.set_ylabel("Radar VV Sesudah (dB)", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("sebaran_fitur_gangguan_hutan.png", dpi=300)
plt.show()
print("Grafik eksplorasi disimpan ke 'sebaran_fitur_gangguan_hutan.png'.")

### 6.3 Pembagian Data dan Pelatihan Model

- **Split Terstrata (Stratified Split 80:20)**: Memastikan proporsi kelas 50% : 30% : 20% tetap identik di data latih (360 baris) dan data uji (90 baris).
- **Model Utama**: `RandomForestClassifier` (100 pohon, kedalaman maksimum 6, `random_state=42`).
- **Model Pembanding**: `DummyClassifier` (selalu menebak kelas mayoritas yaitu *Pertanian*).

In [ ]:
feature_cols = ['ndvi_sebelum', 'ndvi_sesudah', 'radar_vv_sebelum_db', 'radar_vv_sesudah_db', 'luas_hektare']
X = df_forest[feature_cols]
y = df_forest['kelas_gangguan']

# Pembagian data terstrata 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f"Jumlah Data Latih: {len(X_train)} sampel")
print(f"Jumlah Data Uji  : {len(X_test)} sampel")
print("\nDistribusi Kelas pada Data Uji:")
print(y_test.value_counts())

# Inisialisasi dan pelatihan model Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED)
rf_model.fit(X_train, y_train)

# Inisialisasi dan pelatihan model Baseline Dummy
dummy_model = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy_model.fit(X_train, y_train)

print("\nModel Random Forest dan Baseline Dummy Classifier berhasil dilatih.")

### 6.4 Evaluasi Model dan Analisis Metrik

Menghitung akurasi, presisi, recall, dan F1-score untuk data uji, serta membandingkan performa Random Forest terhadap baseline Dummy.

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_pred_dummy = dummy_model.predict(X_test)

# Metrik global
acc_rf = accuracy_score(y_test, y_pred_rf)
acc_dummy = accuracy_score(y_test, y_pred_dummy)
macro_f1_rf = f1_score(y_test, y_pred_rf, average='macro')
weighted_f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print("=" * 65)
print("PERBANDINGAN HASIL EVALUASI MODEL (DATA UJI - 90 SAMPEL)")
print("=" * 65)
print(f"Akurasi Random Forest          : {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print(f"Akurasi Baseline Dummy         : {acc_dummy:.4f} ({acc_dummy*100:.2f}%)")
print(f"Macro Average F1-Score (RF)    : {macro_f1_rf:.4f}")
print(f"Weighted Average F1-Score (RF) : {weighted_f1_rf:.4f}")
print("=" * 65)

target_labels = ['Pertanian', 'Pertambangan', 'Gangguan Alam']
print("\nLaporan Klasifikasi Rinci (Classification Report):")
print(classification_report(y_test, y_pred_rf, target_names=target_labels, digits=4))

# Ekspor tabel evaluasi ke CSV
prec_per_class = precision_score(y_test, y_pred_rf, labels=target_labels, average=None)
rec_per_class = recall_score(y_test, y_pred_rf, labels=target_labels, average=None)
f1_per_class = f1_score(y_test, y_pred_rf, labels=target_labels, average=None)

df_eval = pd.DataFrame({
    'Kelas': target_labels,
    'Precision': np.round(prec_per_class, 4),
    'Recall': np.round(rec_per_class, 4),
    'F1_Score': np.round(f1_per_class, 4)
})
df_eval.loc[len(df_eval)] = ['Macro Average', round(macro_f1_rf, 4), round(macro_f1_rf, 4), round(macro_f1_rf, 4)]
df_eval.loc[len(df_eval)] = ['Overall Accuracy', round(acc_rf, 4), round(acc_rf, 4), round(acc_rf, 4)]

df_eval.to_csv("evaluasi_klasifikasi_hutan.csv", index=False)
print("Tabel metrik disimpan ke 'evaluasi_klasifikasi_hutan.csv'.")

### 6.5 Matriks Kebingungan (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf, labels=target_labels)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_labels)
disp.plot(cmap=plt.cm.Greens, ax=ax, colorbar=False)

ax.set_title("Confusion Matrix: Random Forest (Data Uji)", fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix_hutan.png", dpi=300)
plt.show()
print("Confusion Matrix disimpan ke 'confusion_matrix_hutan.png'.")

### 6.6 Analisis Kepentingan Fitur (*Feature Importance*)

Melihat kontribusi relatif masing-masing fitur biofisik dalam pembuatan keputusan pohon ensemble.

In [ ]:
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({
    'Fitur': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_imp['Fitur'], df_imp['Importance'], color='#2c3e50', edgecolor='black', height=0.55)
ax.set_title("Tingkat Kepentingan Fitur (Random Forest Feature Importance)", fontweight='bold')
ax.set_xlabel("Gini Importance", fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5, axis='x')

for b in ax.patches:
    w = b.get_width()
    ax.annotate(f"{w*100:.1f}%",
                xy=(w, b.get_y() + b.get_height() / 2),
                xytext=(5, 0), textcoords="offset points",
                ha='left', va='center', fontsize=9, fontweight='bold')

plt.xlim(0, max(df_imp['Importance']) * 1.2)
plt.tight_layout()
plt.show()

## 7. Pembahasan Hasil Analisis

### 1. Evaluasi Kinerja Model dan Peran Baseline
- Model Random Forest mencapai akurasi uji tinggi ($> 85\%$) dan Macro F1 yang memuaskan ($> 0.85$), jauh mengungguli Dummy Baseline ($50.00\%$) yang hanya menebak kelas mayoritas *Pertanian*.
- Fitur `ndvi_sesudah` dan `luas_hektare` menjadi prediktor paling dominan. Pertambangan dicirikan oleh penurunan NDVI yang sangat ekstrem (tanah gundul) dan area yang terkonsentrasi besar, sementara pertanian memiliki luasan lebih fleksibel dengan sisa tutupan biomassa moderat.
- Gangguan alam (misalnya banjir atau longsor) memperlihatkan kebingungan klasifikasi parsial terhadap pertanian karena keduanya memiliki variasi kelembapan dan nilai hamburan balik radar yang tumpang tindih.

### 2. Pengaruh Ketidakseimbangan Kelas (Class Imbalance)
- Ketidakseimbangan proporsi kelas ($50\% : 30\% : 20\%$) menyebabkan metrik akurasi global dapat bias menguntungkan kelas mayoritas (*Pertanian*).
- Penggunaan **Macro F1-Score** memberikan pembobotan yang setara kepada setiap kelas tanpa memedulikan ukuran sampelnya. Hal ini memastikan bahwa performa pada kelas minoritas (*Gangguan Alam*) dinilai sama pentingnya dengan kelas mayoritas.

### 3. Batasan Teknis dan Etis AI dalam Pemantauan Hutan
- **Keterbatasan Data Sintetis**: Hubungan statistik yang dibangkitkan hanyalah aproksimasi teoritis sederhana. Pada citra satelit riil, interferensi atmosfer, bayangan awan tebal di hutan tropis, kelembapan tanah musiman, dan heterogenitas kanopi menciptakan derau yang jauh lebih kompleks.
- **Kebutuhan Ground Truth Lapangan**: Prediksi berbasis model satelit merupakan hipotesis awal untuk memandu patroli, bukan kebenaran mutlak. Verifikasi langsung di lapangan (*field-check / ground truthing*) tetap wajib dilakukan.
- **Ketidakmampuan Menentukan Legalitas Hukum**: Model kecerdasan buatan hanya mendeteksi karakteristik biofisik perubahan vegetasi di atas tanah. Model **tidak dapat dan tidak boleh** digunakan untuk menetapkan status legalitas (apakah suatu penebangan berizin sah, konsesi hutan tanaman industri, atau perambahan ilegal) serta motif sosial-ekonomi masyarakat lokal.

## 8. Keterbatasan Studi Kasus

1. **Bukan Citra Satelit Piksel / Raster Asli**: Studi kasus ini menggunakan data tabular buatan, bukan data array citra satelit mentah multispektral (Sentinel-2) atau SLC/GRD (Sentinel-1).
2. **Tidak Menggunakan Arsitektur Spasial-Kontekstual (CNN)**: Model tidak memperhitungkan tekstur spasial sekitar patch gangguan, bentuk geometri garis (seperti pola jalan logging), atau histori runtun waktu (*time-series*).
3. **Penyederhanaan Kategori**: Kategori penyebab gangguan di dunia nyata mencakup penebangan selektif (*selective logging*), kebakaran liar vs kebakaran terencana, infrastruktur jalan, dan perambahan skala kecil yang sangat rumit untuk dipisahkan.

## 9. Kesimpulan

1. Atribusi penyebab gangguan hutan (*driver attribution*) merupakan tahapan analisis lanjutan yang krusial setelah deteksi hilangnya tutupan pohon terkonfirmasi.
2. Kombinasi fitur optik (NDVI) dan radar SAR (VV polarisasi) bersama estimasi luasan area memberikan informasi komplementer yang efektif untuk membedakan indikasi pertanian, pertambangan, dan gangguan alam.
3. Evaluasi menggunakan Macro F1 menjamin penilaian objektif yang melindungi kelas minoritas dari bias akurasi kelas mayoritas.
4. Output model AI harus diposisikan sebagai alat bantu peringatan dini dan prioritas patroli lapangan, bukan vonis hukum sepihak mengenai legalitas aktivitas lahan.